<a href="https://colab.research.google.com/github/anshikaguptacs24-del/AI_LAB/blob/main/1BF24CS058_Week5_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Evaluation of the specific Knowledge Base

**Sentences in KB:**
1. $Q \implies P$
2. $P \implies \neg Q$
3. $Q \lor R$

**Queries to verify:**
- **Query 1**: $R$
- **Query 2**: $R \implies P$
- **Query 3**: $Q \implies R$

In [2]:
import pandas as pd
import itertools

# Variables present in the formulas
variables = ['P', 'Q', 'R']
truth_values = [True, False]
models = list(itertools.product(truth_values, repeat=len(variables)))

def implication(a, b):
    return not a or b

rows = []
for model in models:
    env = dict(zip(variables, model))
    p, q, r = env['P'], env['Q'], env['R']

    # KB Sentences
    s1 = implication(q, p)       # Q -> P
    s2 = implication(p, not q)   # P -> ~Q
    s3 = q or r                  # Q v R

    kb_true = s1 and s2 and s3

    # Queries
    q1 = r                       # R
    q2 = implication(r, p)       # R -> P
    q3 = implication(q, r)       # Q -> R

    rows.append({
        'P': p, 'Q': q, 'R': r,
        'Q -> P': s1,
        'P -> ~Q': s2,
        'Q v R': s3,
        'KB True': kb_true,
        'Query: R': q1,
        'Query: R -> P': q2,
        'Query: Q -> R': q3
    })

df_kb = pd.DataFrame(rows)
display(df_kb)

# Filter models where KB is True
kb_models = df_kb[df_kb['KB True'] == True]
print("Models where KB is True:")
display(kb_models[['P', 'Q', 'R']])

# Check Entailment
print("\n--- Entailment Results ---")
for q_col in ['Query: R', 'Query: R -> P', 'Query: Q -> R']:
    # Entailment holds if every row where 'KB True' is True also has the Query as True
    entails = kb_models[q_col].all() if not kb_models.empty else True
    print(f"Does KB entail '{q_col.replace('Query: ', '')}'? {entails}")

,P,Q,R,Q -> P,P -> ~Q,Q v R,KB True,Query: R,Query: R -> P,Query: Q -> R
0,True,True,True,True,False,True,False,True,True,True
1,True,True,False,True,False,True,False,False,True,False
2,True,False,True,True,True,True,True,True,True,True
3,True,False,False,True,True,False,False,False,True,True
4,False,True,True,False,True,True,False,True,False,True
5,False,True,False,False,True,True,False,False,True,False
6,False,False,True,True,True,True,True,True,False,True
7,False,False,False,True,True,False,False,False,True,True


Models where KB is True:


,P,Q,R
2,True,False,True
6,False,False,True



--- Entailment Results ---
Does KB entail 'R'? True
Does KB entail 'R -> P'? False
Does KB entail 'Q -> R'? True


In [ ]:

import pandas as pd
import itertools
import re

# Take variables as user input (commas or spaces are accepted)
variables = re.split(r"[\s,]+", input(
    "Enter variables separated by commas or spaces (e.g., P,Q,R): "
).strip().upper())

variables = [v for v in variables if v]

if not variables or len(set(variables)) != len(variables):
    raise ValueError("Enter at least one unique variable.")

models = list(itertools.product([True, False], repeat=len(variables)))


def implication(a, b):
    return not a or b


# Evaluate logical expressions safely
def evaluate(expression, env):
    tokens = re.findall(
        r"->|[()~^v]|[A-Za-z_]\w*",
        expression.replace(" ", "")
    )

    if "".join(tokens) != expression.replace(" ", ""):
        raise ValueError(f"Invalid expression: {expression}")

    pos = 0

    def parse_atom():
        nonlocal pos

        if pos >= len(tokens):
            raise ValueError("Incomplete expression")

        token = tokens[pos]

        if token == "~":
            pos += 1
            return not parse_atom()

        if token == "(":
            pos += 1
            value = parse_implication()

            if pos >= len(tokens) or tokens[pos] != ")":
                raise ValueError("Missing closing parenthesis")

            pos += 1
            return value

        if token in env:
            pos += 1
            return env[token]

        raise ValueError(f"Unknown variable: {token}")

    def parse_and():
        nonlocal pos
        value = parse_atom()

        while pos < len(tokens) and tokens[pos] == "^":
            pos += 1
            right = parse_atom()
            value = value and right

        return value

    def parse_or():
        nonlocal pos
        value = parse_and()

        while pos < len(tokens) and tokens[pos] == "v":
            pos += 1
            right = parse_and()
            value = value or right

        return value

    def parse_implication():
        nonlocal pos
        left = parse_or()

        if pos < len(tokens) and tokens[pos] == "->":
            pos += 1
            right = parse_implication()
            return implication(left, right)

        return left

    result = parse_implication()

    if pos != len(tokens):
        raise ValueError(f"Unexpected token: {tokens[pos]}")

    return result


# Input KB sentences
n = int(input("Enter the number of KB sentences: "))
kb_sentences = []

print("Operators: ~ NOT, v OR, ^ AND, -> implication")

for i in range(n):
    kb_sentences.append(input(f"Enter KB sentence {i + 1}: "))

# Input queries
m = int(input("\nEnter the number of queries: "))
queries = []

for i in range(m):
    queries.append(input(f"Enter query {i + 1}: "))

# Generate truth table
rows = []

for model in models:
    env = dict(zip(variables, model))
    kb_values = [evaluate(s, env) for s in kb_sentences]
    kb_true = all(kb_values)

    row = dict(env)

    for i, sentence in enumerate(kb_sentences):
        row[f"KB {i + 1}: {sentence}"] = kb_values[i]

    row["KB True"] = kb_true

    for query in queries:
        row[f"Query: {query}"] = evaluate(query, env)

    rows.append(row)

df_kb = pd.DataFrame(rows)

print("\n--- Truth Table ---")
display(df_kb)

# Find models satisfying the KB
kb_models = df_kb[df_kb["KB True"]]

print("\n--- Models Where KB Is True ---")
display(kb_models[variables])

# Check entailment
print("\n--- Entailment Results ---")

if kb_models.empty:
    print("KB is inconsistent: no models satisfy all KB sentences.")
else:
    for query in queries:
        entails = kb_models[f"Query: {query}"].all()
        print(f"Does KB entail '{query}'? {entails}")
